# E4 · Revenue by nation (a notebook that also runs on a schedule)

This notebook computes revenue per customer nation from the sample orders and saves it as a
table. It runs in **two places** without changes:

| | In your workspace | In Airflow (papermill) |
|---|---|---|
| who runs it | you, cell by cell | the scheduler, top to bottom |
| identity | **you** (your login token) | **lab-batch** (the batch service identity) |
| writes to | your namespace, `eng_<you>` | the shared `analytics` namespace, `u_<you>_...` |

Run it here first (step 2 of the lesson), then schedule it (step 4).

## Parameters

Papermill replaces the values in the next cell (it is tagged `parameters`) with the ones the
DAG passes. Left empty, `target` means "my own namespace".

In [ ]:
target = ""            # table to write; "" = lakehouse.eng_<you>.revenue_by_nation
run_id = "interactive"
dag_id = ""

## Who am I, and where am I running?

In Airflow, the DAG passes a token of the batch identity in `LAB_BATCH_TOKEN`. In your
workspace there is no such variable, so the notebook uses your own login. The notebook never
prints a token.

In [ ]:
import os
import re

if os.environ.get("LAB_BATCH_TOKEN"):
    import trino

    conn = trino.dbapi.connect(
        host=os.environ["LAB_TRINO_HOST"], port=int(os.environ["LAB_TRINO_PORT"]),
        http_scheme="https", verify=os.environ.get("SSL_CERT_FILE", True),
        auth=trino.auth.JWTAuthentication(os.environ["LAB_BATCH_TOKEN"]),
        user=os.environ["LAB_BATCH_PRINCIPAL"], catalog="lakehouse")
    where = "Airflow (papermill)"
else:
    from lakehouse import trino_connection

    conn = trino_connection()
    where = "your workspace"

cur = conn.cursor()
cur.execute("SELECT current_user")
me = cur.fetchone()[0]
if not target and where != "your workspace":
    raise ValueError("no target table: the DAG must pass one, e.g. -p target lakehouse.analytics.u_<you>_revenue_by_nation")
if not target:
    target = f"lakehouse.eng_{re.sub(r'[^a-z0-9_]', '_', me.lower())}.revenue_by_nation"
    cur.execute(f"CREATE SCHEMA IF NOT EXISTS {target.rsplit('.', 1)[0]}")
    cur.fetchall()
print(f"running in {where} as {me}; writing {target}")

## The analysis

Revenue per nation: orders joined to their customer, and the customer to their nation.

In [ ]:
import pandas as pd

QUERY = """
    SELECT n.name                    AS nation,
           count(*)                  AS orders,
           round(sum(o.totalprice), 2) AS revenue
    FROM lakehouse.samples.orders   AS o
    JOIN lakehouse.samples.customer AS c ON c.custkey = o.custkey
    JOIN lakehouse.samples.nation   AS n ON n.nationkey = c.nationkey
    GROUP BY n.name"""

cur.execute(QUERY + " ORDER BY revenue DESC")
df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
print(len(df), "nations")
df.head(10)

## Save it as a table

`CREATE OR REPLACE TABLE ... AS` swaps in the new result in one commit: readers see the old
table or the new one, never a half-written one, and running the notebook twice gives the
same table. The extra columns record **which run** wrote the rows and **as whom**.

In [ ]:
cur.execute(f"""
    CREATE OR REPLACE TABLE {target} AS
    SELECT nation, orders, revenue,
           CAST(? AS varchar) AS run_id,
           CAST(? AS varchar) AS dag_id,
           'trino (notebook)' AS engine,
           current_user       AS written_by,
           current_timestamp(6) AS computed_at
    FROM ({QUERY})""", [run_id, dag_id])
cur.fetchall()
cur.execute(f"SELECT count(*), sum(orders) FROM {target}")
rows, orders = cur.fetchone()
print(f"{target}: {rows} rows, {orders} orders")
assert rows == 25, "every one of the 25 nations should have orders"